# Task 3.4: Data Augmentation - Iteration 2

**Project:** Potato Quality Classification (Fresh vs. Rotten)  
**ILO:** 8.2 C & D  

This notebook applies data augmentation to the training pipeline and retrains the same CNN architecture from Iteration 1 to measure the isolated effect of augmentation.

- **A. Preprocessing** - Same as Iteration 1 + data augmentation on training set only
- **B. CNN Architecture** - Reuse cnn_v1 architecture (3 conv blocks, 32-64-128 filters)
- **C. Training** - Train with augmented data pipeline
- **D. Evaluation** - Compare against Iteration 1 (no augmentation)

---

**Results so far:**
| Model | Accuracy |
|---|---|
| Random Guess | 50.18% |
| MLP Baseline | 79.03% |
| CNN v1 (no augmentation) | 94.35% |

---
## 1. Imports & Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, BatchNormalization,
    Dropout, Flatten, Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

---
## 2. Data Loading with Augmentation

### What changes from Iteration 1?
In Iteration 1, the training generator only rescaled pixel values to [0, 1]. Now we add augmentation transforms to the **training set only**. The validation and test sets remain unchanged so we can fairly evaluate performance.

### Augmentation choices
| Transform | Value | Rationale |
|---|---|---|
| Horizontal flip | True | Potatoes look the same mirrored |
| Rotation | 20 degrees | Slight rotation simulates different camera angles |
| Zoom | 15% | Simulates varying distance from camera |
| Width/height shift | 10% | Simulates potato not being perfectly centered |
| Brightness | [0.8, 1.2] | Simulates different lighting conditions |

These transforms are applied **randomly** each time an image is loaded during training. The model never sees the exact same image twice, which helps prevent overfitting.

In [ ]:
# -- Paths --
BASE_PATH  = '../Task 3.1 Create an Image Dataset ILO 6.2 a en b/data/processed'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH   = os.path.join(BASE_PATH, 'val')
TEST_PATH  = os.path.join(BASE_PATH, 'test')

IMG_SIZE   = (128, 128)
BATCH_SIZE = 32

# -- Training generator WITH augmentation --
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# -- Validation and test generators WITHOUT augmentation --
val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_NAMES = list(train_gen.class_indices.keys())
print('Classes      :', train_gen.class_indices)
print('Train samples:', train_gen.samples)
print('Val samples  :', val_gen.samples)
print('Test samples :', test_gen.samples)

### Visualise augmented vs original images
Below we show the same original image alongside several augmented versions to demonstrate the effect of each transform.

In [ ]:
# Show augmented samples side by side
# Load one original image for comparison
orig_datagen = ImageDataGenerator(rescale=1.0 / 255)
orig_gen = orig_datagen.flow_from_directory(
    TRAIN_PATH, target_size=IMG_SIZE, batch_size=1,
    class_mode='categorical', shuffle=False, seed=SEED
)
aug_gen = train_datagen.flow_from_directory(
    TRAIN_PATH, target_size=IMG_SIZE, batch_size=1,
    class_mode='categorical', shuffle=False, seed=SEED
)

# Get one original image
orig_img, orig_label = next(orig_gen)

# Generate 7 augmented versions of the same batch index
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes[0][0].imshow(orig_img[0])
axes[0][0].set_title('Original', fontsize=10, fontweight='bold')
axes[0][0].axis('off')

for i, ax in enumerate(axes.flat[1:]):
    aug_img, _ = next(aug_gen)
    ax.imshow(aug_img[0])
    ax.set_title(f'Augmented {i+1}', fontsize=10)
    ax.axis('off')

plt.suptitle('Original vs Augmented Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. CNN Architecture - cnn_v2

We reuse the **exact same architecture** as Iteration 1 (3 conv blocks: 32-64-128 filters, Dense head with 256 units). This way, any difference in performance can be attributed purely to data augmentation, not architectural changes.

In [ ]:
def build_cnn_v2(input_shape=(128, 128, 3), n_classes=2):
    """Same architecture as cnn_v1 - only the data pipeline changes."""
    model = Sequential([
        Input(shape=input_shape),

        # -- Convolutional Block 1 --
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # -- Convolutional Block 2 --
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # -- Convolutional Block 3 --
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # -- Classifier Head --
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(n_classes, activation='softmax')
    ], name='cnn_v2')
    return model


cnn_v2 = build_cnn_v2()
cnn_v2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_v2.summary()

---
## 4. Training with Early Stopping

Same training setup as Iteration 1: Adam optimizer (lr=0.001), early stopping with patience=5. The only difference is that the training data now comes through the augmented pipeline.

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history_v2 = cnn_v2.fit(
    train_gen,
    epochs=50,
    validation_data=val_gen,
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTraining stopped after {len(history_v2.history["loss"])} epoch(s).')

---
## 5. Learning Curves

In [ ]:
def plot_learning_curves(history, title_suffix=''):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'],     label='Train loss')
    axes[0].plot(history.history['val_loss'], label='Val loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Loss {title_suffix}')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'],     label='Train accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Val accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'Accuracy {title_suffix}')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_learning_curves(history_v2, title_suffix='- CNN v2 (with augmentation)')

---
## 6. Evaluation on the Test Set

In [ ]:
# -- Predict on test set --
test_gen.reset()
y_pred_proba = cnn_v2.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes

# -- Metrics --
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted')
rec  = recall_score(y_true, y_pred, average='weighted')
f1   = f1_score(y_true, y_pred, average='weighted')

print('=' * 45)
print(f'  Accuracy  : {acc  * 100:.2f} %')
print(f'  Precision : {prec * 100:.2f} %')
print(f'  Recall    : {rec  * 100:.2f} %')
print(f'  F1 Score  : {f1   * 100:.2f} %')
print('=' * 45)

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# -- Confusion matrix heatmap --
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - CNN v2 (Test Set)')
plt.tight_layout()
plt.show()

---
## 7. Comparison: Iteration 1 vs Iteration 2

We compare all models to see whether data augmentation improved performance.

In [ ]:
RANDOM_GUESS_ACC = 50.18
MLP_BASELINE_ACC = 79.03
CNN_V1_ACC       = 94.35   # from Iteration 1
cnn_v2_acc       = acc * 100

models   = ['Random Guess', 'MLP Baseline', 'CNN v1\n(no aug)', 'CNN v2\n(with aug)']
acc_vals = [RANDOM_GUESS_ACC, MLP_BASELINE_ACC, CNN_V1_ACC, cnn_v2_acc]
colours  = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, acc_vals, color=colours, edgecolor='black', width=0.5)

for bar, val in zip(bars, acc_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{val:.2f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison - All Iterations', fontsize=13)
plt.tight_layout()
plt.show()

print('\n-- Comparison Table --')
print(f'  Random Guess  : {RANDOM_GUESS_ACC:.2f}%')
print(f'  MLP Baseline  : {MLP_BASELINE_ACC:.2f}%')
print(f'  CNN v1 (no aug): {CNN_V1_ACC:.2f}%')
print(f'  CNN v2 (aug)   : {cnn_v2_acc:.2f}%')
delta = cnn_v2_acc - CNN_V1_ACC
print(f'\n  Effect of augmentation: {delta:+.2f} percentage points')

---
## 8. Summary & Conclusions Iteration 2

**What changed:** Only the training data pipeline. Data augmentation (horizontal flip, rotation up to 20 degrees, zoom up to 15%, width/height shift up to 10%, brightness range 0.8-1.2) was added to the training generator. The CNN architecture, optimizer, and all hyperparameters remained identical to Iteration 1.

**Results:**
- CNN v1 (no augmentation): 94.35%
- CNN v2 (with augmentation): 93.28%
- Effect: -1.07 percentage points

**Analysis:** Data augmentation slightly decreased performance by 1.07 percentage points. This is likely because the dataset (3,712 images from 3 diverse sources) already provides sufficient variety for the model to generalize well. The augmentation transforms made the training data harder to learn from without providing additional useful variation, as evidenced by the very high initial validation loss (15.87 in epoch 1) compared to Iteration 1. The model still performs well above the MLP baseline (79.03%), confirming that the CNN architecture is effective for this task.

**Next steps for Iteration 3:** Transfer learning using a pretrained model (e.g., ResNet50 or VGG16) to leverage features learned on ImageNet.